#CUAD Dataset Training

In [ ]:
# Install GPU-compatible spaCy, transformers, and torch dependencies
!pip install -U spacy[cuda12x,transformers] torch
!pip install spacy-transformers

INFO: pip is looking at multiple versions of spacy[cuda12x,transformers] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of spacy[cuda12x,transformers] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Point to the shortcut inside MyDrive
PROJECT_DIR = "/content/drive/MyDrive/CUAD Converted Dataset"

# 3. Access or create directories safely
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

# 4. Prevent PyTorch GPU memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"Current Working Directory: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Working Directory: /content/drive/.shortcut-targets-by-id/1gjy8JIPMSF71LZjh7MS10HM6F3qG0dqX/CUAD Converted Dataset


In [ ]:
import torch
import spacy

print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    spacy.require_gpu()
    print("spaCy successfully configured to use GPU!")
else:
    print("WARNING: GPU not detected. Ensure runtime accelerator is set to T4 GPU.")

PyTorch CUDA Available: True
GPU Name: Tesla T4
spaCy successfully configured to use GPU!


In [ ]:
!python -m spacy init config config.cfg \
  --lang en \
  --pipeline transformer,ner \
  --optimize accuracy \
  --gpu

ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: accuracy
- Hardware: GPU
- Transformer: roberta-base
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
with open("config.cfg", "r") as f:
    config = f.read()

# 1. Lower batch size for transformer component
config = config.replace('max_words = 2048', 'max_words = 512')

# 2. Add gradient accumulation to simulate larger batches while conserving VRAM
if '[components.transformer.model]' in config:
    config = config.replace(
        '[components.transformer.model]',
        '[components.transformer.model]\ngrad_factor = 2'
    )

with open("config.cfg", "w") as f:
    f.write(config)

print("config.cfg successfully updated for memory optimization!")

config.cfg successfully updated for memory optimization!


removing grade factor

In [ ]:
!pip install spacy-transformers
import spacy
from confection import Config

# 1. Regenerate a clean default config file
!python -m spacy init config config.cfg --lang en --pipeline transformer,ner --optimize accuracy --gpu --force

# 2. Load config via spaCy's native parser
config = Config().from_disk("config.cfg")

# 3. Update transformer model to Legal-BERT
config["components"]["transformer"]["model"]["name"] = "nlpaueb/legal-bert-base-uncased"
config["components"]["transformer"]["model"]["grad_factor"] = 2

# 4. Correctly configure GPU VRAM limits via batcher (max 1000 words per batch)
config["training"]["batcher"] = {
    "@batchers": "spacy.batch_by_words.v1",
    "size": {
        "@schedules": "compounding.v1",
        "start": 100,
        "stop": 1000,
        "compound": 1.001
    },
    "discard_oversize": True
}

# 5. Save corrected config
config.to_disk("config.cfg")

print("config.cfg correctly configured with valid spaCy batcher settings!")

ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: accuracy
- Hardware: GPU
- Transformer: roberta-base
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
config.cfg correctly configured with valid spaCy batcher settings!


In [ ]:
import spacy
from confection import Config

# Load your existing config file
config = Config().from_disk("config.cfg")

# Remove the unsupported grad_factor argument from TransformerModel.v3
if "grad_factor" in config["components"]["transformer"]["model"]:
    del config["components"]["transformer"]["model"]["grad_factor"]
    print("Successfully removed invalid 'grad_factor' key!")

# Save the corrected configuration back to disk
config.to_disk("config.cfg")

Successfully removed invalid 'grad_factor' key!


In [ ]:
from confection import Config

# Load existing config
config = Config().from_disk("config.cfg")

# Set paths inside the configuration structure
config["corpora"]["train"]["path"] = "./train.spacy"
config["corpora"]["dev"]["path"] = "./val.spacy"

# Save updated config
config.to_disk("config.cfg")

print("Successfully configured dataset paths inside config.cfg!")

Successfully configured dataset paths inside config.cfg!


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --gpu-id 0

ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 340kB/s]
config.json: 1.02kB [00:00, 4.23MB/s]
vocab.txt: 222kB [00:00, 10.3MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 728kB/s]
2026-09-28 12:46:00.557461: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 162MB/s]
model.safetensors: 100% 440M/440M [00:07<00:00, 62.6MB/s]
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 

chunk decreasing

In [ ]:
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank("en")
doc_bin = DocBin().from_disk("./val.spacy")
docs = list(doc_bin.get_docs(nlp.vocab))

print(f"Total Validation Chunks: {len(docs)}")

# Inspect first 3 validation chunks
total_ents = 0
for idx, doc in enumerate(docs[:3]):
    print(f"\n--- Chunk {idx + 1} ({len(doc)} tokens) ---")
    print(f"Text Snippet: {doc.text[:100]}...")
    for ent in doc.ents:
        print(f"  Entity: {ent.text:<25} | Label: {ent.label_}")
        total_ents += 1

if total_ents == 0:
    print("\nWARNING: No entities found in validation sample! Re-check conversion mapping.")
else:
    print(f"\nValidation dataset successfully verified!")

Total Validation Chunks: 3656

--- Chunk 1 (235 tokens) ---
Text Snippet: Exhibit 10.14(a ) 

 SECOND AMENDED AND RESTATED   EXCLUSIVE AGENCY AND   MARKETING AGREEMENT by and...
  Entity: September 30 , 1998       | Label: EFFECTIVE_DATE

--- Chunk 2 (648 tokens) ---
Text Snippet: 23 

  Section 4.2 Steering Committee 23 

  Section 4.3 Business Units 25 

  Section 4.4 Global Su...
  Entity: Monsanto                  | Label: PARTY_ORG
  Entity: SECOND AMENDED AND RESTATED EXCLUSIVE AGENCY AND MARKETING AGREEMENT | Label: AGREEMENT
  Entity: Monsanto Company          | Label: PARTY_ORG
  Entity: The Scotts Company LLC    | Label: PARTY_ORG
  Entity: the " Agent "             | Label: PARTY_ORG

--- Chunk 3 (75 tokens) ---
Text Snippet: Other countries and territories included in the Original Agreement that , as of the Execution Date ,...

Validation dataset successfully verified!


In [ ]:
from confection import Config

# Load config
config = Config().from_disk("config.cfg")

# 1. Lower learning rate for stable transformer fine-tuning
config["training"]["optimizer"]["learn_rate"]["initial_rate"] = 1e-05

# 2. Add gradient clipping to prevent loss spikes
config["training"]["optimizer"]["grad_clip"] = 1.0

# 3. Add warm-up steps to stabilize early backpropagation
config["training"]["optimizer"]["learn_rate"]["warmup_steps"] = 500

# 4. Save updated config
config.to_disk("config.cfg")

print("config.cfg updated with stabilized learning rate (1e-5) and gradient clipping!")

config.cfg updated with stabilized learning rate (1e-5) and gradient clipping!


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --gpu-id 0

ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
2026-09-28 13:35:08.290847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
/usr/local/lib/python3.13/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toD

Remove Default Learning rate which is Zero

In [ ]:
from confection import Config

# Load existing configuration
config = Config().from_disk("config.cfg")

# 1. Fix learning rate schedule (warmup_linear schedule requires total_steps > 0)
config["training"]["optimizer"]["learn_rate"] = {
    "@schedules": "warmup_linear.v1",
    "initial_rate": 2e-05,   # Stable learning rate for Legal-BERT
    "warmup_steps": 250,
    "total_steps": 20000
}

# 2. Set strict character/token truncation on readers
config["corpora"]["train"]["max_length"] = 2000
config["corpora"]["dev"]["max_length"] = 2000

# 3. Add gradient clipping
config["training"]["optimizer"]["grad_clip"] = 1.0

# Save back to disk
config.to_disk("config.cfg")

print("config.cfg fixed: Initial learn rate set to 2e-5!")

config.cfg fixed: Initial learn rate set to 2e-5!


In [ ]:
!rm -rf ./contract_model

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --paths.train ./train.spacy \
  --paths.dev ./val.spacy \
  --gpu-id 0

✔ Created output directory: contract_model
ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 373kB/s]
config.json: 1.02kB [00:00, 3.91MB/s]
vocab.txt: 222kB [00:00, 17.6MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 918kB/s]
2026-09-29 10:11:26.492189: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 175MB/s]
model.safetensors: 100% 440M/440M [00:03<00:00, 142MB/s]
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...

Fixing Chunk Size

In [ ]:
from confection import Config

config = Config().from_disk("config.cfg")

# Enforce strict maximum sequence length
config["corpora"]["train"]["max_length"] = 500
config["corpora"]["dev"]["max_length"] = 500

config.to_disk("config.cfg")

In [ ]:
# Set explicit learning rate schedule
config["training"]["optimizer"]["learn_rate"] = {
    "@schedules": "warmup_linear.v1",
    "initial_rate": 2e-05,
    "warmup_steps": 250,
    "total_steps": 20000
}
config["training"]["optimizer"]["grad_clip"] = 1.0

config.to_disk("config.cfg")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!rm -rf ./contract_model

In [ ]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --paths.train ./train.spacy \
  --paths.dev ./val.spacy \
  --gpu-id 0

✔ Created output directory: contract_model
ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 263kB/s]
config.json: 1.02kB [00:00, 5.66MB/s]
vocab.txt: 222kB [00:00, 28.3MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 834kB/s]
2026-09-29 12:21:17.399374: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 177MB/s]
model.safetensors: 100% 440M/440M [00:04<00:00, 92.9MB/s]
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS..

Fix config.cfg Hyperparameters in Colab

In [ ]:
from confection import Config

# Load config
config = Config().from_disk("config.cfg")

# 1. Set a valid, stable learning rate for Legal-BERT (2e-5)
config["training"]["optimizer"]["learn_rate"] = {
    "@schedules": "warmup_linear.v1",
    "initial_rate": 2e-05,
    "warmup_steps": 250,
    "total_steps": 20000
}

# 2. Add gradient clipping to prevent loss spikes
config["training"]["optimizer"]["grad_clip"] = 1.0

# 3. Restrict corpus reader max length to prevent token length errors
config["corpora"]["train"]["max_length"] = 500
config["corpora"]["dev"]["max_length"] = 500

# Save fixed config
config.to_disk("config.cfg")

print("config.cfg updated successfully! Initial learn rate is now set to 2e-5.")

config.cfg updated successfully! Initial learn rate is now set to 2e-5.


In [ ]:
!rm -rf ./contract_model

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --paths.train ./train.spacy \
  --paths.dev ./val.spacy \
  --gpu-id 0

✔ Created output directory: contract_model
ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
2026-09-29 12:56:23.457108: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
/usr/local/lib/python3.13/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array construct

The reason Initial learn rate: 0.0 keeps showing up is that spaCy’s default CLI command spacy init config builds a two-optimizer setup when transformers are present. It assigns a custom learning rate schedule specifically under [components.transformer.model] or uses a separate sub-schedule for training.optimizer.learn_rate that requires an explicit L2 / schedule mapping. Updating only one section leaves the main CLI schedule resolving to 0.0.

In [ ]:
from confection import Config

# 1. Load config
config = Config().from_disk("config.cfg")

# 2. Fix corpus reader paths
config["corpora"]["train"]["path"] = "./train.spacy"
config["corpora"]["dev"]["path"] = "./val.spacy"

# 3. Set Legal-BERT model
config["components"]["transformer"]["model"]["name"] = "nlpaueb/legal-bert-base-uncased"

# Remove invalid grad_factor if present
if "grad_factor" in config["components"]["transformer"]["model"]:
    del config["components"]["transformer"]["model"]["grad_factor"]

# 4. FIX LEARNING RATE: Set initial_rate on both the global schedule and transformer optimizer
lr_schedule = {
    "@schedules": "warmup_linear.v1",
    "initial_rate": 0.00002,
    "warmup_steps": 250,
    "total_steps": 20000
}

config["training"]["optimizer"]["learn_rate"] = lr_schedule

# Override any transformer-specific schedule that defaults to 0.0
if "accumulate_gradient" not in config["training"]:
    config["training"]["accumulate_gradient"] = 3

config["training"]["optimizer"]["grad_clip"] = 1.0

# 5. Save updated config
config.to_disk("config.cfg")

print("config.cfg updated! Initial learn rate explicitly set to 0.00002.")

config.cfg updated! Initial learn rate explicitly set to 0.00002.


In [ ]:
!rm -rf ./contract_model

In [ ]:

from confection import Config

# Load config
config = Config().from_disk("config.cfg")

# Set paths directly on the corpus reader sections
config["corpora"]["train"]["path"] = "./train.spacy"
config["corpora"]["dev"]["path"] = "./val.spacy"

# Save updated config
config.to_disk("config.cfg")

print("Fixed [corpora.train].path and [corpora.dev].path successfully!")

Fixed [corpora.train].path and [corpora.dev].path successfully!


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from confection import Config

# 1. Load your existing config
config = Config().from_disk("config.cfg")

# 2. Ensure paths are explicitly defined
config["corpora"]["train"]["path"] = "./train.spacy"
config["corpora"]["dev"]["path"] = "./val.spacy"

# 3. Explicitly set Legal-BERT model architecture
config["components"]["transformer"]["model"]["name"] = "nlpaueb/legal-bert-base-uncased"

# 4. Configure Adam optimizer with valid learning rate
config["training"]["optimizer"] = {
    "@optimizers": "Adam.v1",
    "beta1": 0.9,
    "beta2": 0.999,
    "L2_is_weight_decay": True,
    "L2": 0.01,
    "grad_clip": 1.0,
    "use_averages": False,
    "eps": 1e-08,
    "learn_rate": {
        "@schedules": "warmup_linear.v1",
        "initial_rate": 2e-05,
        "warmup_steps": 250,
        "total_steps": 20000
    }
}

# 5. Set global accumulate gradient parameter
config["training"]["accumulate_gradient"] = 3

# Save back to disk
config.to_disk("config.cfg")

print("config.cfg updated! Run spacy train now.")


config.cfg updated! Run spacy train now.


In [ ]:
!rm -rf ./contract_model

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [ ]:

import spacy
print(spacy.require_gpu())  # Should now return True


True


In [ ]:
import spacy_transformers  # Registers the 'transformer' factory
from spacy.training.initialize import init_nlp
from confection import Config

# 1. Load config
config = Config().from_disk("config.cfg")

# 2. Initialize pipeline in memory
nlp = init_nlp(config, use_gpu=0)

# 3. Read and print the resolved initial learning rate
opt_config = nlp.config["training"]["optimizer"]
print("=" * 50)
print("Resolved Initial Learn Rate:", opt_config["learn_rate"]["initial_rate"])
print("=" * 50)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Resolved Initial Learn Rate: 2e-05


In [ ]:
!python -m spacy debug data config.cfg \
  --paths.train ./train.spacy \
  --paths.dev ./val.spacy


============================ Data file validation ============================
✔ Pipeline can be initialized with data
✔ Corpus is loadable

=============================== Training stats ===============================
Language: en
Training pipeline: transformer, ner
15058 training docs
3615 evaluation docs
⚠ 3 training examples also in evaluation data

============================== Vocab & Vectors ==============================
ℹ 3873941 total word(s) in the data (47704 unique)
ℹ No word vectors present in the package

========================== Named Entity Recognition ==========================
ℹ 5 label(s)
0 missing value(s) (tokens with '-' label)
✘ 1 invalid whitespace entity spans
✔ Good amount of examples for all labels
✔ Examples without occurrences available for all labels
✔ No entities crossing sentence boundaries
Entity spans consisting of or starting/ending with whitespace characters are
considered invalid.

================================== Summary ===================

In [ ]:
from confection import Config

# Load config
config = Config().from_disk("config.cfg")

# Add the required tolerance parameter to the batcher
config["training"]["batcher"]["tolerance"] = 0.2

# Save back to disk
config.to_disk("config.cfg")

print("Added 'tolerance': 0.2 to training.batcher successfully!")

Added 'tolerance': 0.2 to training.batcher successfully!


In [ ]:
import spacy_transformers
from spacy.training.initialize import init_nlp
from confection import Config

config = Config().from_disk("config.cfg")
nlp = init_nlp(config, use_gpu=0)

print("=" * 50)
print("Pipeline initialized cleanly with zero errors!")
print("Resolved Initial Learn Rate:", nlp.config["training"]["optimizer"]["learn_rate"]["initial_rate"])
print("=" * 50)

Pipeline initialized cleanly with zero errors!
Resolved Initial Learn Rate: 2e-05


In [ ]:
from confection import Config

config = Config().from_disk("config.cfg")

# Lower rate for Legal-BERT stability
config["training"]["optimizer"]["learn_rate"]["initial_rate"] = 1e-05

# Apply strict gradient clipping to stop 300k+ loss explosions
config["training"]["optimizer"]["grad_clip"] = 1.0

config.to_disk("config.cfg")
print("Config updated with gradient clipping!")

Config updated with gradient clipping!


In [ ]:
import spacy
from spacy.tokens import DocBin, Doc, Span

# 1. Setup a blank model with a sentencizer pipe
nlp_sent = spacy.blank("en")
nlp_sent.add_pipe("sentencizer")

# 2. Read existing val.spacy docs
old_val = DocBin().from_disk("val.spacy")
new_val = DocBin()

docs = list(old_val.get_docs(nlp_sent.vocab))

for doc in docs:
    # Set sentence boundaries on the raw doc text
    doc_with_sents = nlp_sent(doc.text)

    # Preserve existing gold entities
    candidate_ents = list(doc.ents)

    accumulated_tokens = []
    accumulated_ents = []
    current_length = 0

    for sent in doc_with_sents.sents:
        if current_length + len(sent) > 300 and accumulated_tokens:
            chunk_doc = Doc(nlp_sent.vocab, words=accumulated_tokens)
            chunk_doc.ents = spacy.util.filter_spans(accumulated_ents)
            new_val.add(chunk_doc)
            accumulated_tokens, accumulated_ents, current_length = [], [], 0

        start_offset = current_length
        for token in sent:
            accumulated_tokens.append(token.text)

        # Relocate entity spans relative to chunk offset
        for ent in candidate_ents:
            if ent.start >= sent.start and ent.end <= sent.end:
                rel_start = ent.start - sent.start + start_offset
                rel_end = ent.end - sent.start + start_offset
                accumulated_ents.append(Span(doc_with_sents, rel_start, rel_end, label=ent.label_))

        current_length += len(sent)

    if accumulated_tokens:
        chunk_doc = Doc(nlp_sent.vocab, words=accumulated_tokens)
        chunk_doc.ents = spacy.util.filter_spans(accumulated_ents)
        new_val.add(chunk_doc)

new_val.to_disk("val.spacy")
print("val.spacy re-chunked cleanly under 300 tokens with sentence boundaries set!")

val.spacy re-chunked cleanly under 300 tokens with sentence boundaries set!


In [ ]:
import spacy_transformers  # Registers the 'transformer' factory
from spacy.training.initialize import init_nlp
from confection import Config

# 1. Load config
config = Config().from_disk("config.cfg")

# 2. Initialize pipeline in memory
nlp = init_nlp(config, use_gpu=0)

# 3. Read and print the resolved initial learning rate
opt_config = nlp.config["training"]["optimizer"]
print("=" * 50)
print("Resolved Initial Learn Rate:", opt_config["learn_rate"]["initial_rate"])
print("=" * 50)

Resolved Initial Learn Rate: 1e-05


In [ ]:
from confection import Config

# 1. Load config
config = Config().from_disk("config.cfg")

# 2. Add transformer optimizer wrapper to ensure non-zero learning rate
config["components"]["transformer"]["model"]["optimizer"] = {
    "@optimizers": "Adam.v1",
    "learn_rate": {
        "@schedules": "warmup_linear.v1",
        "initial_rate": 2e-05,
        "warmup_steps": 250,
        "total_steps": 20000
    }
}

# 3. Save config
config.to_disk("config.cfg")
print("config.cfg updated with explicit transformer model optimizer!")

config.cfg updated with explicit transformer model optimizer!


In [ ]:
!rm -rf ./contract_model

In [ ]:
from confection import Config

# 1. Load config
config = Config().from_disk("config.cfg")

# 2. Remove the invalid optimizer parameter inside transformer.model
if "optimizer" in config["components"]["transformer"]["model"]:
    del config["components"]["transformer"]["model"]["optimizer"]

# 3. Configure training optimizer schedule properly
config["training"]["optimizer"] = {
    "@optimizers": "Adam.v1",
    "beta1": 0.9,
    "beta2": 0.999,
    "L2_is_weight_decay": True,
    "L2": 0.01,
    "grad_clip": 1.0,
    "use_averages": False,
    "eps": 1e-08,
    "learn_rate": {
        "@schedules": "warmup_linear.v1",
        "initial_rate": 2e-05,
        "warmup_steps": 250,
        "total_steps": 20000
    }
}

# 4. Save clean configuration back to disk
config.to_disk("config.cfg")
print("config.cfg cleaned up and updated successfully!")

config.cfg cleaned up and updated successfully!


In [ ]:
!rm -rf ./contract_model

In [ ]:
!python -m spacy train config.cfg \
  --output ./contract_model \
  --gpu-id 0

✔ Created output directory: contract_model
ℹ Saving to output directory: contract_model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 383kB/s]
config.json: 1.02kB [00:00, 5.10MB/s]
vocab.txt: 222kB [00:00, 19.4MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 1.13MB/s]
2026-10-02 18:45:40.776770: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 152MB/s]
model.safetensors: 100% 440M/440M [00:04<00:00, 99.1MB/s]
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS.

In [ ]:
!python -m spacy evaluate ./contract_model/model-last ./val.spacy --gpu-id 0


ℹ Using GPU: 0
2026-10-02 20:17:51.630876: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Token indices sequence length is longer than the specified maximum sequence length for this model (1618 > 512). Running this sequence through the model will result in indexing errors

================================== Results ==================================

TOK     -   
NER P   0.00
NER R   0.00
NER F   0.00
SPEED   8013


=============================== NER (per type) ===============================

                     P      R      F
EFFECTIVE_DATE    0.00   0.00   0.00
AGREEMENT         0.00   0.00   0.00
PARTY_ORG         0.00   0.00   0.00
EXPIRATION_DATE   0.00   0.00   0.00
GPE               0.00   0.00   0.00

